# IaC-Eval Intent Audit Results

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

plt.style.use('ggplot')
sns.set_palette("muted")

BENCHMARK_CSV = "../data/iac_eval_benchmark.csv"

## IaC-Eval's own pass criterion (v2 audit, 15 Oct 2026)

In [2]:
import glob, re
V2_REF = "result/final/rego_reference_ceiling/v2"
V2_GLM = "result/final/rego_iaceval_glm_v2"

def load_v2(folder, main_pat, *rerun_pats):
    main = pd.concat([pd.read_csv(f) for f in sorted(glob.glob(f"{folder}/{main_pat}"))])
    for rerun_pat in rerun_pats:
        rr = [pd.read_csv(f) for f in sorted(glob.glob(f"{folder}/{rerun_pat}"))]
        if not rr:
            continue
        rr = pd.concat(rr)
        main = pd.concat([main[~main.row_number.isin(rr.row_number)], rr])
    main = main.sort_values("row_number").reset_index(drop=True)
    main["ie_pass"] = main["rego_passed_iaceval"].astype(str).eq("True")
    main["old_pass"] = main["rego_passed"].astype(str).eq("True")
    return main

ref = load_v2(V2_REF, "Rego_reference_part*.csv", "Rego_reference_rerun*.csv", "Rego_reference_rrB*.csv")
ref = ref.merge(pd.read_csv("result/final/rego_reference_ceiling/IaCEval_reference_outputs_relaxed.csv")[["row_number", "required_version_removed"]], on="row_number")
glm = load_v2(V2_GLM, "Rego_glm_part*.csv", "Rego_glm_rerun*.csv", "Rego_glm_rrB*.csv")
assert ref.row_number.nunique() == 458 and glm.row_number.nunique() == 458
ref.to_csv("result/final/rego_reference_ceiling/Rego_reference_relaxed_v2_final.csv", index=False)
glm.to_csv("result/final/Rego_TF_IaCEval_GLM53F_PlanOnly_v2_final.csv", index=False)

ENV_RE = re.compile(r"timed out after|Failed to query available provider packages|Failed to install provider|no such host|registry\.terraform\.io|releases\.hashicorp\.com")
def status(r):
    e = r.terraform_error
    if pd.isna(e) or e == "":
        if pd.notna(r.opa_error) and "undefined function" in str(r.opa_error): return "policy error (benchmark)"
        return "pass" if r.ie_pass else "planned, policy not met"
    e = str(e)
    if "no available releases match the given constraints" in e: return "version drift: module needs a newer provider"
    if ENV_RE.search(e): return "environment: provider download"
    if "Archive creation error" in e: return "dataset: missing local file"
    if "query returned no results" in e: return "emulator: data-source lookup"
    if "Unsupported argument" in e or "Unsupported block type" in e: return "provider schema drift"
    return "other Terraform error"
ref["status"] = ref.apply(status, axis=1)
glm["status"] = glm.apply(status, axis=1)
print("Reference status:"); display(ref.status.value_counts().to_frame("references"))
print("GLM status:"); display(glm.status.value_counts().to_frame("templates"))

Reference status:


,references
status,
pass,292
"planned, policy not met",55
version drift: module needs a newer provider,47
provider schema drift,25
dataset: missing local file,21
emulator: data-source lookup,13
other Terraform error,5


GLM status:


,templates
status,
"planned, policy not met",245
pass,212
policy error (benchmark),1


In [3]:
bench = pd.read_csv(BENCHMARK_CSV)[["row_number", "difficulty"]]
both = glm[["row_number", "ie_pass", "old_pass", "status"]].merge(
    ref[["row_number", "ie_pass", "status", "required_version_removed"]], on="row_number", suffixes=("_glm", "_ref")).merge(bench, on="row_number")
ref_ok = both.ie_pass_ref
glm_env = both.status_glm.str.startswith("environment")
summary = pd.DataFrame([
    {"set": "all 458 tasks", "N": len(both), "reference passes (%)": 100 * both.ie_pass_ref.mean(), "GLM passes (%)": 100 * both.ie_pass_glm.mean()},
    {"set": "tasks whose reference passes", "N": int(ref_ok.sum()), "reference passes (%)": 100.0, "GLM passes (%)": 100 * both.loc[ref_ok, "ie_pass_glm"].mean()},
    {"set": "reference passes, reference unpinned", "N": int((ref_ok & ~both.required_version_removed).sum()), "reference passes (%)": 100.0,
     "GLM passes (%)": 100 * both.loc[ref_ok & ~both.required_version_removed, "ie_pass_glm"].mean()},
]).round(1)
display(summary)
print(f"GLM templates with an environment error left: {int(glm_env.sum())}")
print(f"Old criterion (is_configuration_valid only): GLM {int(both.old_pass.sum())}/458; under IaC-Eval's criterion: {int(both.ie_pass_glm.sum())}/458")
print(f"Strict as-is ceiling (pinned references fail on Terraform 1.14): {int((both.ie_pass_ref & ~both.required_version_removed).sum())}/458")
by_level = both.groupby("difficulty")[["ie_pass_ref","ie_pass_glm"]].apply(lambda g: pd.Series({
    "N": len(g), "reference pass %": 100 * g.ie_pass_ref.mean(), "GLM pass %": 100 * g.ie_pass_glm.mean(),
    "N (reference passes)": int(g.ie_pass_ref.sum()), "GLM pass % | reference passes": 100 * g.loc[g.ie_pass_ref, "ie_pass_glm"].mean() if g.ie_pass_ref.any() else float("nan")})).round(1)
display(by_level)
sat = pd.read_csv("result/final/iac_eval_satisfiability_audit.csv")[["row_number", "is_label_dependent"]]
lab = both.merge(sat, on="row_number")
display(lab.groupby("is_label_dependent")[["ie_pass_ref","ie_pass_glm"]].apply(lambda g: pd.Series({"N": len(g), "reference pass %": 100 * g.ie_pass_ref.mean(),
        "GLM pass %": 100 * g.ie_pass_glm.mean(), "GLM pass % | reference passes": 100 * g.loc[g.ie_pass_ref, "ie_pass_glm"].mean()})).round(1))

,set,N,reference passes (%),GLM passes (%)
0,all 458 tasks,458,63.8,46.3
1,tasks whose reference passes,292,100.0,63.0
2,"reference passes, reference unpinned",197,100.0,66.5


GLM templates with an environment error left: 0
Old criterion (is_configuration_valid only): GLM 11/458; under IaC-Eval's criterion: 212/458
Strict as-is ceiling (pinned references fail on Terraform 1.14): 197/458


,N,reference pass %,GLM pass %,N (reference passes),GLM pass % | reference passes
difficulty,,,,,
1,45.0,97.8,64.4,44.0,65.9
2,95.0,84.2,61.1,80.0,65.0
3,113.0,64.6,50.4,73.0,68.5
4,58.0,62.1,48.3,36.0,58.3
5,72.0,37.5,30.6,27.0,55.6
6,75.0,42.7,24.0,32.0,53.1


,N,reference pass %,GLM pass %,GLM pass % | reference passes
is_label_dependent,,,,
False,403.0,66.5,51.1,67.2
True,55.0,43.6,10.9,16.7


In [4]:
import json as _json
def plan_types(folder, rn):
    f = Path(folder) / "plans" / f"row_{rn}.json"
    if not f.exists(): return None
    pj = _json.load(open(f))
    out = []
    def walk(mod):
        for r in mod.get("resources", []) or []:
            if r.get("mode") == "managed": out.append(r["type"])
        for c in mod.get("child_modules", []) or []: walk(c)
    walk((pj.get("planned_values") or {}).get("root_module") or {})
    return out

rows = []
for rn in both.loc[both.ie_pass_ref, "row_number"]:
    g, r = plan_types(V2_GLM, rn), plan_types(V2_REF, rn)
    if g is None or r is None: continue
    gs, rs = set(g), set(r)
    gr = glm.loc[glm.row_number == rn].iloc[0]
    false_rules = [x for x in str(gr.iaceval_false_rules).split(";") if x and x != "nan"]
    rows.append({"row_number": rn, "glm_pass": bool(gr.ie_pass), "n_false_rules": len(false_rules),
                 "glm_resources": len(g), "ref_resources": len(r), "extra_types": len(gs - rs), "missing_types": len(rs - gs),
                 "extra_type_list": ";".join(sorted(gs - rs)), "missing_type_list": ";".join(sorted(rs - gs))})
cmp_df = pd.DataFrame(rows)
cmp_df.to_csv("result/final/Rego_TF_IaCEval_GLM53F_PlanOnly_v2_vs_reference.csv", index=False)
print(f"tasks compared (reference passes, both planned): {len(cmp_df)}")
display(cmp_df.groupby("glm_pass")[["glm_resources", "ref_resources", "extra_types", "missing_types", "n_false_rules"]].mean().round(2))
fails = cmp_df[~cmp_df.glm_pass]
print(f"GLM failures on these tasks: {len(fails)} | with >=1 missing reference type: {int((fails.missing_types > 0).sum())} "
      f"| with extra types only (no missing): {int(((fails.extra_types > 0) & (fails.missing_types == 0)).sum())} "
      f"| same type set: {int(((fails.extra_types == 0) & (fails.missing_types == 0)).sum())}")
from collections import Counter
print("Most frequent extra types in GLM failures:", Counter(t for s in fails.extra_type_list for t in s.split(";") if t).most_common(10))
print("Most frequent missing types in GLM failures:", Counter(t for s in fails.missing_type_list for t in s.split(";") if t).most_common(10))
fr = Counter()
for s in glm.loc[glm.row_number.isin(fails.row_number), "iaceval_false_rules"].dropna():
    for x in str(s).split(";"):
        if x: fr[x.split(".")[-1]] += 1
print("Most frequent false rules:", fr.most_common(12))

tasks compared (reference passes, both planned): 292


,glm_resources,ref_resources,extra_types,missing_types,n_false_rules
glm_pass,,,,,
False,12.45,3.95,6.56,0.60,1.47
True,9.66,3.59,5.29,0.45,0.00


GLM failures on these tasks: 108 | with >=1 missing reference type: 41 | with extra types only (no missing): 57 | same type set: 10
Most frequent extra types in GLM failures: [('aws_route_table', 32), ('aws_route_table_association', 32), ('aws_subnet', 32), ('aws_vpc', 32), ('aws_kms_key', 31), ('aws_security_group', 30), ('aws_s3_bucket_server_side_encryption_configuration', 30), ('aws_s3_bucket_public_access_block', 29), ('aws_secretsmanager_secret', 28), ('aws_s3_bucket_versioning', 28)]
Most frequent missing types in GLM failures: [('aws_lex_bot', 5), ('aws_lex_intent', 5), ('aws_s3_object', 4), ('aws_vpc_security_group_egress_rule', 4), ('aws_vpc_security_group_ingress_rule', 4), ('random_string', 3), ('aws_elastic_beanstalk_configuration_template', 2), ('aws_iam_policy', 2), ('aws_iam_role_policy_attachment', 2), ('aws_elasticache_cluster', 2)]
Most frequent false rules: [('is_configuration_valid', 24), ('allow', 17), ('valid', 16), ('has_valid_resources', 13), ('is_valid_db_inst

In [5]:
runs = pd.read_csv("../results/TF_IaCEval_GLM53F_PlanOnly/results_final.csv", engine="python", usecols=["row_number", "iterations_used"])
it = both.merge(runs, on="row_number")
it["iterations"] = it.iterations_used.clip(upper=4).map(lambda v: f"{int(v)}" if v < 4 else "4+")
display(it[it.ie_pass_ref].groupby("iterations").agg(N=("row_number", "size"), glm_intent_pass_pct=("ie_pass_glm", lambda s: round(100 * s.mean(), 1))))

,N,glm_intent_pass_pct
iterations,,
1,121,62.8
2,106,66.0
3,52,59.6
4+,13,53.8
